In [1]:
## Import modules
import os,sys
import numpy as np
import geopandas as gpd
import cftime 
import gc
import shapely
import json 
import logging
import glob
import datetime
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
from pathlib import Path
import psutil
import warnings
warnings.filterwarnings('ignore')

# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Import utilities for this comparison
sys.path.insert(0,cmct_dir)
from cmct.time_utils import check_datarange
from cmct.calving import *
from cmct.calving_modules.interpolation import *
from cmct.calving_modules.json_to_netcdf import *
from cmct.shapefile_utils import *          

# Force initial garbage collection
gc.collect()

# Configure logging to only show errors
logging.basicConfig(level=logging.ERROR, format='%(asctime)s - %(levelname)s - %(message)s')

# Memory monitoring function
def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def clear_memory():
    """Aggressive memory clearing"""
    gc.collect()
    print(f"Memory after cleanup: {get_memory_usage():.1f} MB")

In [2]:
# Reload modules to pick up any changes to imports
import importlib
import cmct.calving
import cmct.shapefile_utils
importlib.reload(cmct.shapefile_utils)
importlib.reload(cmct.calving)

# Re-import to ensure functions are available
from cmct.calving import *
from cmct.shapefile_utils import *

### Configuration - Ensemble Version

In [3]:
# Observation Dataset
# Ice sheet
loc = 'GIS' # 'GIS' or 'AIS'

# Set the observation data dir path
obs_filename = cmct_dir + '/data/calving/observed_icemask_ismip_annual.nc'

# To use aggregation functions for basin 
basin_aggregation = True # IMPORTANT

basin_shapes = cmct_dir + '/data/ne_10m_coastline/ne_10m_coastline.shp'

# ENSEMBLE CONFIGURATION - Multiple model files using glob
# Define the model file template pattern
model_filename_template = cmct_dir + '/test/calving/ensemble/*.nc'

# Find all model files using glob
model_files = glob.glob(model_filename_template)

# Sort the files for consistent ordering
model_files.sort()

# Generate model names from filenames (extract basename without extension)
model_names = [os.path.splitext(os.path.basename(f))[0] for f in model_files]

# Validate that we found model files
if not model_files:
    raise FileNotFoundError(f"No model files found matching pattern: {model_filename_template}")

print(f"Found {len(model_files)} model files:")
for i, (name, file) in enumerate(zip(model_names, model_files), 1):
    print(f"  {i}. {name} -> {os.path.basename(file)}")

# Set time range for comparison
start_year = 2008
end_year = 2012

# List of basins (ex [NW, NE]) to compare if all -> "all"
basin_list = ["NW"]

# Output configuration
output_dir = cmct_dir + '/output/ensemble_results/'
os.makedirs(output_dir, exist_ok=True)

# Chunk processing configuration
chunk_size = 2  # Process 2 years at a time to manage memory
memory_threshold = 8000  # MB - trigger cleanup if memory exceeds this

# Optional Configurations 
interpolation_method = 'nearest' # 'nearest', 'linear', 'cubic'
accuracy_calculation_method = 'mean' # 'mean', 'RMS'

colors = {
    'CW': 'blue',
    'NE': 'red', 
    'SE': 'green',
    'SW': 'orange',
    'NO': 'purple',
    'NW': 'brown'
}

print(f"Ensemble processing configured for {len(model_files)} model files")
print(f"Initial memory usage: {get_memory_usage():.1f} MB")

Found 5 model files:
  1. sftgif_B001_hist -> sftgif_B001_hist.nc
  2. sftgif_B002_hist -> sftgif_B002_hist.nc
  3. sftgif_B003_hist -> sftgif_B003_hist.nc
  4. sftgif_B004_hist -> sftgif_B004_hist.nc
  5. sftgif_B005_hist -> sftgif_B005_hist.nc
Ensemble processing configured for 5 model files
Initial memory usage: 260.5 MB


## Error Checking and Setup

In [4]:
# Check if observation file exists
if not os.path.exists(obs_filename):
    raise FileNotFoundError(f"Observation file not found: {obs_filename}")

# Check if all model files exist
for i, model_file in enumerate(model_files):
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"Model file {i+1} not found: {model_file}")

# Validate model_names length matches model_files
if len(model_names) != len(model_files):
    raise ValueError("Number of model names must match number of model files")

print(f"All {len(model_files)} model files found and validated")
print(f"Model files to process: {model_names}")

All 5 model files found and validated
Model files to process: ['sftgif_B001_hist', 'sftgif_B002_hist', 'sftgif_B003_hist', 'sftgif_B004_hist', 'sftgif_B005_hist']


In [5]:
# Load observation data once (shared across all models)
print("Loading observation data...")
gsfc = load_gsfc_calving(obs_filename)
gsfc.ds["time"] = gsfc.time

# Load basins once (shared across all models)
print("Loading basin data...")
basins, basin_list = load_basins_exp(cmct_dir, basin_list)

print(f"Observation data loaded. Memory usage: {get_memory_usage():.1f} MB")
print(f"Basins loaded: {basin_list}")

Loading observation data...
Loading basin data...
Observation data loaded. Memory usage: 977.2 MB
Basins loaded: ['NW']


# Ensemble Processing Functions

In [6]:
def process_single_model(model_file, model_name, gsfc, basins, basin_list, 
                        start_year, end_year, chunk_size, output_dir):
    """
    Process a single model file with simplified nearest neighbor approach
    Compares model to observations using nearest coordinates, treating NaN as 0
    """
    print(f"\n{'='*60}")
    print(f"Processing model: {model_name}")
    print(f"{'='*60}")
    
    try:
        # Load model data
        model_ds = xr.open_dataset(model_file)
        if 'sftgif' not in model_ds.data_vars:
            print(f"Error: 'sftgif' variable not found in {model_name}")
            model_ds.close()
            return None
        
        # Convert time to years
        time_values = model_ds.time.values
        if hasattr(time_values[0], 'year'):
            years = [dt.year for dt in time_values]
        else:
            years = time_values.astype(int)
        
        # Find overlapping years
        overlap_years = set(years).intersection(set(range(start_year, end_year + 1))).intersection(set(gsfc.ds.year.values))
        if not overlap_years:
            print(f"No overlapping years found for {model_name}")
            model_ds.close()
            return None
        
        print(f"Processing {len(overlap_years)} overlapping years: {sorted(overlap_years)}")
        
        # Get coordinate arrays
        model_x, model_y = model_ds.x.values, model_ds.y.values
        obs_x, obs_y = gsfc.ds.x.values, gsfc.ds.y.values
        
        # Find nearest neighbor mappings using simplified approach
        print("Creating coordinate mappings using nearest neighbor...")
        
        # Create coordinate mapping matrices
        model_x_2d, model_y_2d = np.meshgrid(model_x, model_y)
        model_coords = np.column_stack([model_x_2d.ravel(), model_y_2d.ravel()])
        
        obs_x_2d, obs_y_2d = np.meshgrid(obs_x, obs_y)
        obs_coords = np.column_stack([obs_x_2d.ravel(), obs_y_2d.ravel()])
        
        # Simple nearest neighbor using broadcasting (more efficient than scipy)
        def find_nearest_indices(target_coords, source_coords):
            distances = np.sqrt(((target_coords[:, None, :] - source_coords[None, :, :])**2).sum(axis=2))
            return np.argmin(distances, axis=1)
        
        # Find nearest observation points for each model point
        nearest_obs_indices = find_nearest_indices(model_coords, obs_coords)
        
        # Convert back to 2D indices
        obs_i_indices = nearest_obs_indices // len(obs_x)
        obs_j_indices = nearest_obs_indices % len(obs_x)
        model_i_indices = np.arange(len(model_y))[:, None] * np.ones(len(model_x), dtype=int)
        model_j_indices = np.ones(len(model_y), dtype=int)[:, None] * np.arange(len(model_x))
        
        print(f"Mapped {len(model_x) * len(model_y)} model points to nearest observation points")
        
        # Process years in chunks
        all_analyses = []
        processing_years = sorted(overlap_years)
        
        for year in processing_years:
            try:
                # Get data for this year
                year_idx = list(years).index(year)
                model_data = model_ds['sftgif'].isel(time=year_idx).values
                obs_data = gsfc.ds.sel(year=year)['ice_mask'].values
                
                # Replace NaN with 0
                model_data = np.where(np.isnan(model_data), 0, model_data)
                obs_data = np.where(np.isnan(obs_data), 0, obs_data)
                
                # Extract comparison data using nearest neighbor mapping
                model_flat = model_data.ravel()
                obs_nearest = obs_data[obs_i_indices, obs_j_indices]
                
                # Calculate residuals and statistics
                residuals = obs_nearest - model_flat
                
                analysis = {
                    'year': year,
                    'comparison_points': len(residuals),
                    'mean_residual': float(np.mean(residuals)),
                    'rms_residual': float(np.sqrt(np.mean(residuals**2))),
                    'min_residual': float(np.min(residuals)),
                    'max_residual': float(np.max(residuals)),
                    'agreement_count': int(np.sum((model_flat == 0) == (obs_nearest == 0))),
                    'agreement_percent': float(np.mean((model_flat == 0) == (obs_nearest == 0)) * 100),
                    'model_mean': float(np.mean(model_flat)),
                    'obs_mean': float(np.mean(obs_nearest))
                }
                
                all_analyses.append(analysis)
                print(f"  Year {year}: {len(residuals)} points, mean_res={analysis['mean_residual']:.4f}, "
                      f"RMS={analysis['rms_residual']:.4f}, agreement={analysis['agreement_percent']:.1f}%")
                
            except Exception as e:
                print(f"Error processing year {year}: {e}")
                continue
        
        # Create basin statistics using residual data
        stats = {}
        for basin in basin_list:
            stats[basin] = {}
            for analysis in all_analyses:
                year = analysis['year']
                stats[basin][year] = {
                    'mean': analysis['mean_residual'],
                    'rms': analysis['rms_residual'],
                    'comparison_points': analysis['comparison_points'],
                    'agreement_percent': analysis['agreement_percent'],
                    'min_residual': analysis['min_residual'],
                    'max_residual': analysis['max_residual'],
                    'model_mean': analysis['model_mean'],
                    'obs_mean': analysis['obs_mean']
                }
        
        # Save results
        stats_file = f"{output_dir}/{model_name}_comparison_stats.json"
        with open(stats_file, 'w') as f:
            json.dump({
                'model_name': model_name,
                'basin_stats': stats,
                'summary_analyses': all_analyses,
                'processed_years': processing_years
            }, f, indent=2, default=str)
        
        model_ds.close()
        gc.collect()
        
        print(f"Model {model_name} processing complete. Results saved.")
        
        return {
            'model_name': model_name,
            'basin_stats': stats,
            'analyses': all_analyses
        }
        
    except Exception as e:
        print(f"Error processing {model_name}: {e}")
        return None

# Ensemble Processing Loop

In [7]:
# Initialize ensemble results storage
ensemble_results = {}
ensemble_stats = {}

print(f"Starting ensemble processing of {len(model_files)} models...")
print(f"Processing years: {start_year} to {end_year}")
print(f"Output directory: {output_dir}")

# Process each model
for i, (model_file, model_name) in enumerate(zip(model_files, model_names)):
    print(f"\n[{i+1}/{len(model_files)}] Processing {model_name}...")
    
    try:
        # Process single model
        model_result = process_single_model(
            model_file, model_name, gsfc, basins, basin_list,
            start_year, end_year, chunk_size, output_dir
        )
        
        if model_result:
            ensemble_results[model_name] = model_result
            ensemble_stats[model_name] = model_result['basin_stats']
            
            # Print summary for this model
            print(f"\nSummary for {model_name}:")
            format_basin_statistics(basin_list, model_result['basin_stats'])
            
        # Aggressive cleanup between models
        clear_memory()
        
    except Exception as e:
        print(f"Error processing {model_name}: {e}")
        continue

print(f"\nEnsemble processing complete!")
print(f"Successfully processed {len(ensemble_results)} out of {len(model_files)} models")
print(f"Results saved in: {output_dir}")

Starting ensemble processing of 5 models...
Processing years: 2008 to 2012
Output directory: /Users/aditya_pachpande/Documents/GitHub/CmCt/output/ensemble_results/

[1/5] Processing sftgif_B001_hist...

Processing model: sftgif_B001_hist
Processing 5 overlapping years: [2008, 2009, 2010, 2011, 2012]
Creating coordinate mappings using nearest neighbor...
Error processing sftgif_B001_hist: Unable to allocate 341. TiB for an array with shape (4842961, 4838400, 2) and data type float64
Memory after cleanup: 1147.7 MB

[2/5] Processing sftgif_B002_hist...

Processing model: sftgif_B002_hist
Processing 5 overlapping years: [2008, 2009, 2010, 2011, 2012]
Creating coordinate mappings using nearest neighbor...
Error processing sftgif_B002_hist: Unable to allocate 341. TiB for an array with shape (4842961, 4838400, 2) and data type float64
Memory after cleanup: 1124.7 MB

[3/5] Processing sftgif_B003_hist...

Processing model: sftgif_B003_hist
Processing 5 overlapping years: [2008, 2009, 2010, 2

# Ensemble Analysis and Visualization

In [8]:
def create_ensemble_comparison_plots(ensemble_stats, basin_list, colors, output_dir):
    """
    Create comparison plots for all models in the ensemble using residual-based metrics
    """
    if not ensemble_stats:
        print("No ensemble statistics available for plotting")
        return
    
    # Create subplots for each basin
    n_basins = len(basin_list)
    fig, axes = plt.subplots(n_basins, 3, figsize=(18, 4*n_basins))
    
    if n_basins == 1:
        axes = axes.reshape(1, -1)
    
    for i, basin in enumerate(basin_list):
        ax_mean = axes[i, 0]
        ax_rms = axes[i, 1]
        ax_agreement = axes[i, 2]
        
        # Plot residual statistics for all models
        for model_name, stats in ensemble_stats.items():
            if basin in stats:
                basin_data = stats[basin]
                years = list(basin_data.keys())
                mean_residuals = [basin_data[year]['mean'] for year in years]
                rms_residuals = [basin_data[year]['rms'] for year in years]
                agreement_pct = [basin_data[year].get('agreement_percent', 0) for year in years]
                
                ax_mean.plot(years, mean_residuals, label=model_name, 
                           color=colors.get(basin, 'black'), 
                           linestyle='--' if 'MODEL' in model_name else '-',
                           marker='o', markersize=4)
                
                ax_rms.plot(years, rms_residuals, label=model_name,
                          color=colors.get(basin, 'black'),
                          linestyle='--' if 'MODEL' in model_name else '-',
                          marker='s', markersize=4)
                
                ax_agreement.plot(years, agreement_pct, label=model_name,
                                color=colors.get(basin, 'black'),
                                linestyle='--' if 'MODEL' in model_name else '-',
                                marker='^', markersize=4)
        
        ax_mean.set_title(f'{basin} Basin - Mean Residuals (Obs - Model)')
        ax_mean.set_xlabel('Year')
        ax_mean.set_ylabel('Mean Residual')
        ax_mean.legend()
        ax_mean.grid(True, alpha=0.3)
        ax_mean.axhline(y=0, color='red', linestyle='--', alpha=0.5)
        
        ax_rms.set_title(f'{basin} Basin - RMS Residuals')
        ax_rms.set_xlabel('Year')
        ax_rms.set_ylabel('RMS Residual')
        ax_rms.legend()
        ax_rms.grid(True, alpha=0.3)
        
        ax_agreement.set_title(f'{basin} Basin - Agreement Percentage')
        ax_agreement.set_xlabel('Year')
        ax_agreement.set_ylabel('Agreement (%)')
        ax_agreement.legend()
        ax_agreement.grid(True, alpha=0.3)
        ax_agreement.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/ensemble_comparison_residuals.png', dpi=300, bbox_inches='tight')
    plt.show()

# Create ensemble comparison plots
create_ensemble_comparison_plots(ensemble_stats, basin_list, colors, output_dir)

No ensemble statistics available for plotting


In [9]:
def calculate_ensemble_statistics(ensemble_stats, basin_list):
    """
    Calculate ensemble mean, std, min, max for each basin and year
    """
    ensemble_summary = {}
    
    for basin in basin_list:
        ensemble_summary[basin] = {}
        
        # Get all years from first model
        first_model = list(ensemble_stats.keys())[0]
        if basin in ensemble_stats[first_model]:
            years = list(ensemble_stats[first_model][basin].keys())
            
            for year in years:
                mean_values = []
                rms_values = []
                
                for model_name, stats in ensemble_stats.items():
                    if basin in stats and year in stats[basin]:
                        mean_values.append(stats[basin][year]['mean'])
                        rms_values.append(stats[basin][year]['rms'])
                
                if mean_values and rms_values:
                    ensemble_summary[basin][year] = {
                        'mean': {
                            'ensemble_mean': np.mean(mean_values),
                            'ensemble_std': np.std(mean_values),
                            'ensemble_min': np.min(mean_values),
                            'ensemble_max': np.max(mean_values),
                            'n_models': len(mean_values)
                        },
                        'rms': {
                            'ensemble_mean': np.mean(rms_values),
                            'ensemble_std': np.std(rms_values),
                            'ensemble_min': np.min(rms_values),
                            'ensemble_max': np.max(rms_values),
                            'n_models': len(rms_values)
                        }
                    }
    
    return ensemble_summary

# Calculate ensemble statistics
ensemble_summary = calculate_ensemble_statistics(ensemble_stats, basin_list)

# Save ensemble summary
with open(f'{output_dir}/ensemble_summary.json', 'w') as f:
    json.dump(ensemble_summary, f, indent=2, default=str)

print("Ensemble Summary:")
for basin in basin_list:
    print(f"\n{basin} Basin:")
    if basin in ensemble_summary:
        for year in sorted(ensemble_summary[basin].keys()):
            year_data = ensemble_summary[basin][year]
            print(f"  {year}: Mean={year_data['mean']['ensemble_mean']:.4f}±{year_data['mean']['ensemble_std']:.4f}, "
                  f"RMS={year_data['rms']['ensemble_mean']:.4f}±{year_data['rms']['ensemble_std']:.4f} "
                  f"({year_data['mean']['n_models']} models)")

IndexError: list index out of range

# Final Memory Cleanup

In [ ]:
# Final cleanup
if 'gsfc' in locals():
    del gsfc
if 'basins' in locals():
    del basins
if 'ensemble_results' in locals():
    del ensemble_results
if 'ensemble_stats' in locals():
    del ensemble_stats

clear_memory()
print(f"\nProcessing complete! Check {output_dir} for results.")
print(f"Final memory usage: {get_memory_usage():.1f} MB")

Memory after cleanup: 595.5 MB

Processing complete! Check /Users/aditya_pachpande/Documents/GitHub/CmCt/output/ensemble_results/ for results.
Final memory usage: 595.5 MB
